In [ ]:
import pandas as pd
import numpy as np
import csv


accounts = pd.read_csv(
    "../all_csv/accounts.csv",
    sep=",",
    quoting=csv.QUOTE_NONE
)

branches = pd.read_csv(
    "../all_csv/branches.csv",
    sep=",",
    quoting=csv.QUOTE_NONE
)

card = pd.read_csv(
    "../all_csv/cards.csv",
    sep=",",
    quoting=csv.QUOTE_NONE
)

employee = pd.read_csv(
    "../all_csv/employees.csv",
    sep=",",
    quoting=csv.QUOTE_NONE
)

customers = pd.read_csv(
    "../all_csv/customers.csv",
    sep=",",
    quoting=csv.QUOTE_NONE
)

loan_pay = pd.read_csv(
    "../all_csv/loan_payments.csv",
    sep=",",
    quoting=csv.QUOTE_NONE
)

loan = pd.read_csv(
    "../all_csv/loans-_1_.csv",
    sep=",",
    quoting=csv.QUOTE_NONE
)

sup_tic = pd.read_csv(
    "../all_csv/support_tickets-_1_.csv",
    sep=",",
    quoting=csv.QUOTE_NONE
)

tranc = pd.read_csv(
    "../all_csv/transactions-_1_.csv",
    sep=",",
    quoting=csv.QUOTE_NONE
)

card_trans = pd.read_csv(
    "../all_csv/card_transactions.csv",
    sep=",",
    quoting=csv.QUOTE_NONE
)

# — Put all tables into one dictionary
tables = {
    "accounts": accounts,
    "branches": branches,
    "card": card,
    "employee": employee,
    "customers": customers,
    "loan_pay": loan_pay,
    "loan": loan,
    "sup_tic": sup_tic,
    "tranc": tranc,
    "card_trans": card_trans
}

for table in tables.values():


    table.columns = (
        table.columns
        .str.strip()
        .str.strip('"')
    )

    for col in table.select_dtypes(include="str").columns:
        table[col] = (
            table[col]
            .str.strip()
            .str.strip('"')
        )


for name, table in tables.items():

    print(f"\n{'='*20} {name.upper()} {'='*20}")

    print("Shape:", table.shape)

    print("\nColumns:")
    print(table.columns.tolist())

    print("\nData types:")
    print(table.dtypes)

    print("\nNull values:")
    print(table.isna().sum())

    print("\nDuplicate rows:")
    print(table.duplicated().sum())

    print("\nFirst 5 rows:")
    print(table.head())


date_cols = {
    "accounts": ["open_date"],

    "branches": ["opened_date"],

    "card": [
        "issue_date",
        "expiry_date"
    ],

    "customers": [
        "date_of_birth",
        "join_date"
    ],

    "loan_pay": [
        "payment_date"
    ],

    "loan": [
        "start_date"
    ],

    "sup_tic": [
        "date_opened",
        "date_resolved"
    ],

    "tranc": [
        "txn_date"
    ],

    "card_trans": [
        "txn_date"
    ],

    "employee": [
        "hire_date"
    ]
}
for name, cols in date_cols.items():

    for col in cols:

        tables[name][col] = pd.to_datetime(
            tables[name][col],
            errors="coerce"
        )


numeric_cols = {

    "accounts": [
        "account_id",
        "customer_id",
        "branch_id",
        "balance"
    ],

    "branches": [
        "branch_id"
    ],

    "card": [
        "card_id",
        "customer_id",
        "account_id",
        "credit_limit"
    ],

    "employee": [
        "employee_id",
        "branch_id",
        "salary"
    ],

    "customers": [
        "customer_id",
        "credit_score"
    ],

    "loan_pay": [
        "payment_id",
        "loan_id",
        "amount",
        "late_payment_flag"
    ],

    "loan": [
        "loan_id",
        "customer_id",
        "principal_amount",
        "interest_rate"
    ],

    "sup_tic": [
        "ticket_id",
        "customer_id",
        "satisfaction_score"
    ],

    "tranc": [
        "transaction_id",
        "account_id",
        "amount"
    ],

    "card_trans": [
        "card_txn_id",
        "card_id",
        "amount",
        "is_fraud"
    ]
}


for name, cols in numeric_cols.items():

    for col in cols:

        if col in tables[name].columns:

            tables[name][col] = pd.to_numeric(
                tables[name][col],
                errors="coerce"
            )
print("\n----------DATA TYPE FIXED ------------")

for name, table in tables.items():

    print(f"\n----- {name} -----")
    print(table.dtypes)


for name, table in tables.items():

    nulls = table.isna().sum()

    print(f"\n----- {name} -----")
    print(nulls[nulls > 0])
for name, table in tables.items():

    print(
        f"{name}:",
        table.duplicated().sum(),
        "duplicate rows"
    )



primary_keys = {
    "accounts": "account_id",
    "branches": "branch_id",
    "card": "card_id",
    "employee": "employee_id",
    "customers": "customer_id",
    "loan_pay": "payment_id",
    "loan": "loan_id",
    "sup_tic": "ticket_id",
    "tranc": "transaction_id",
    "card_trans": "card_txn_id"
}

for name, key in primary_keys.items():

    table = tables[name]

    print(
        f"{name}.{key}:",
        "rows =", len(table),
        "| unique =", table[key].nunique(),
        "| null =", table[key].isna().sum()
    )


relationships = [

    ("accounts", "customer_id", "customers", "customer_id"),

    ("accounts", "branch_id", "branches", "branch_id"),

    ("card", "customer_id", "customers", "customer_id"),

    ("card", "account_id", "accounts", "account_id"),

    ("employee", "branch_id", "branches", "branch_id"),

    ("loan", "customer_id", "customers", "customer_id"),

    ("loan_pay", "loan_id", "loan", "loan_id"),

    ("sup_tic", "customer_id", "customers", "customer_id"),

    ("tranc", "account_id", "accounts", "account_id"),

    ("card_trans", "card_id", "card", "card_id")
]
for child_table, child_col, parent_table, parent_col in relationships:

    child = tables[child_table][child_col]
    parent = tables[parent_table][parent_col]

    invalid = (~child.isin(parent)).sum()

    print(
        f"\n{child_table}.{child_col}"
        f" → "
        f"{parent_table}.{parent_col}"
        f" | invalid IDs = {invalid}"
    )



category_rules = {

    "accounts": {
        "account_type": [
            "Current",
            "Savings",
            "Salary",
            "Fixed Deposit",
            "NRI"
        ],
        "status": [
            "Active",
            "Dormant",
            "Closed"
        ]
    },

    "card": {
        "card_type": [
            "Credit - Gold",
            "Credit - Classic",
            "Debit",
            "Credit - Platinum"
        ],
        "status": [
            "Active",
            "Expired",
            "Blocked"
        ]
    },

    "employee": {
        "role": [
            "Customer Service",
            "Relationship Manager",
            "Loan Officer",
            "Branch Manager",
            "Teller",
            "Compliance Officer"
        ]
    },

    "customers": {
        "gender": [
            "Male",
            "Female",
            "Other"
        ],
        "occupation": [
            "Retired",
            "Salaried - Government",
            "Salaried - Private",
            "Freelancer",
            "Self Employed",
            "Business Owner",
            "Student"
        ]
    },

    "loan": {
        "loan_type": [
            "Auto Loan",
            "Education Loan",
            "Gold Loan",
            "Home Loan",
            "Personal Loan",
            "Business Loan"
        ],
        "status": [
            "Active",
            "Defaulted",
            "Closed",
            "Written Off"
        ]
    },

    "sup_tic": {
        "issue_type": [
            "App Login Issue",
            "Card Blocked",
            "KYC Update",
            "Interest Query",
            "Cheque Bounce",
            "Account Statement",
            "Net Banking Issue",
            "Loan Query",
            "Wrong Debit",
            "Fraud Report"
        ],
        "status": [
            "Resolved",
            "Open",
            "Escalated"
        ]
    },

    "tranc": {
        "txn_type": [
            "Deposit",
            "Withdrawal",
            "Interest Credit",
            "Fee Debit",
            "Transfer Out",
            "Transfer In"
        ],
        "channel": [
            "Online Banking",
            "Mobile App",
            "UPI",
            "Branch",
            "POS",
            "ATM"
        ]
    }
}
for table_name, columns in category_rules.items():

    table = tables[table_name]

    for col, allowed_values in columns.items():

        invalid = ~table[col].isin(allowed_values)
        print(f"\n===={table_name}====")
        print(
            f"\n{table_name}.{col}:",
            invalid.sum(),
            "invalid categories"
        )


print("Credit score:")
print(customers["credit_score"].describe())

print("\nBalance:")
print(accounts["balance"].describe())

print("\nSalary:")
print(employee["salary"].describe())

print("\nInterest rate:")
print(loan["interest_rate"].describe())

print("\nSatisfaction score:")
print(sup_tic["satisfaction_score"].describe())


print(
    "Invalid credit scores:",
    ((customers["credit_score"] < 0) |
     (customers["credit_score"] > 1000)).sum()
)

print(
    "Invalid satisfaction scores:",
    ((sup_tic["satisfaction_score"] < 1) |
     (sup_tic["satisfaction_score"] > 5)).sum()
)

print(
    "Invalid fraud flags:",
    (~card_trans["is_fraud"].isin([0, 1])).sum()
)

print(
    "Invalid late payment flags:",
    (~loan_pay["late_payment_flag"].isin([0, 1])).sum()
)


print(
    "Invalid card dates:",
    (card["issue_date"] > card["expiry_date"]).sum()
)

print(
    "Invalid ticket dates:",
    (sup_tic["date_resolved"] < sup_tic["date_opened"]).sum()
)


print(
    "Invalid customer dates:",
    (customers["join_date"] < customers["date_of_birth"]).sum()
)


for name, table in tables.items():

    print(f"\n{'='*50}")
    print(f"{name.upper()}")
    print(f"{'='*50}")

    print("Rows:", len(table))
    print("Columns:", len(table.columns))
    print("Duplicate rows:", table.duplicated().sum())
    print("Total null values:", table.isna().sum().sum())

    print("\nData types:")
    print(table.dtypes)




==================== ACCOUNTS ====================
Shape: (95000, 7)

Columns:
['account_id', 'customer_id', 'branch_id', 'account_type', 'balance', 'open_date', 'status']

Data types:
account_id          str
customer_id       int64
branch_id         int64
account_type        str
balance         float64
open_date           str
status              str
dtype: object

Null values:
account_id      0
customer_id     0
branch_id       0
account_type    0
balance         0
open_date       0
status          0
dtype: int64

Duplicate rows:
0

First 5 rows:
  account_id  customer_id  branch_id account_type   balance   open_date  \
0          1        11749        141      Current  95902.00  2019-10-20   
1          2        47716         43      Current  13261.61  2026-02-03   
2          3        56143         90      Savings  26495.92  2020-10-19   
3          4        29193        113       Salary  74682.72  2012-02-22   
4          5        49907         28      Current  40419.01  2007-01-0

In [40]:
#==============================================
# GROUP 1 — CUSTOMER ANALYSIS
#==============================================


# 1 - Occupation-wise Average Annual Income.
occupation_income = (
    customers
    .groupby("occupation", as_index=False)
    .agg(
        average_income=("annual_income", "mean"),
        customer_count=("customer_id", "nunique")
    )
    .sort_values("average_income", ascending=False)
)

print("Q1 — Occupation-wise Average Annual Income")
display(occupation_income)

# 2 - Which occupation groups have the highest average credit score?
occupation_credit = (
    customers
    .groupby("occupation", as_index=False)
    .agg(
        average_credit_score=("credit_score", "mean"),
        customer_count=("customer_id", "nunique")
    )
    .sort_values("average_credit_score", ascending=False)
)

print("\nQ2 — Occupation-wise Average Credit Score")
display(occupation_credit)


# 3 — Customer Distribution by Credit Score Bands

credit_bins = [0, 499, 599, 699, 799, 1000]

credit_labels = [
    "Very Low",
    "Low",
    "Fair",
    "Good",
    "Excellent"
]
customers["credit_score_band"] = pd.cut(
    customers["credit_score"],
    bins=credit_bins,
    labels=credit_labels
)
credit_band_analysis = (
    customers
    .groupby("credit_score_band", observed=True)
    .agg(
        customer_count=("customer_id", "nunique")
    )
    .reset_index()
)
credit_band_analysis["percentage"] = (
    credit_band_analysis["customer_count"]
    / credit_band_analysis["customer_count"].sum()
    * 100
)
print("Q3 — Customer Distribution by Credit Score Bands")
display(credit_band_analysis)


# Q4 — High Income + Low Credit Score Customers

income_threshold = customers["annual_income"].quantile(0.75)
high_income_low_credit = (
    customers[
        (customers["annual_income"] >= income_threshold) &
        (customers["credit_score"] < 600)
    ]
    [
        [
            "customer_id",
            "name",
            "occupation",
            "annual_income",
            "credit_score",
            "city",
            "state"
        ]
    ]
    .sort_values(
        ["credit_score", "annual_income"],
        ascending=[True, False]
    )
)

print("Q4 — High Income + Low Credit Score Customers")
display(high_income_low_credit)


# Q5 — Which States Have the Highest Customer Concentration?
state_customers = (
    customers
    .groupby("state", as_index=False)
    .agg(
        customer_count=("customer_id", "nunique")
    )
    .sort_values("customer_count", ascending=False)
)
state_customers["percentage"] = (
    state_customers["customer_count"]
    / state_customers["customer_count"].sum()
    * 100
)

print("Q5 — State-wise Customer Concentration")
display(state_customers)


# Q6 — Which States Have the Highest Average Annual Income?

state_income = (
    customers
    .groupby("state", as_index=False)
    .agg(
        average_income=("annual_income", "mean"),
        customer_count=("customer_id", "nunique")
    )
    .sort_values("average_income", ascending=False)
)

print("Q6 — State-wise Average Annual Income")
display(state_income)

# Q7 — What Is the Relationship Between Annual Income and Credit Score?

income_credit_corr = (
    customers["annual_income"]
    .corr(customers["credit_score"])
)

print("Q7 — Correlation Between Annual Income and Credit Score")
print(f"Correlation: {income_credit_corr:.3f}")


# Q8 — Which Customers Have Multiple Accounts?

multiple_accounts = (
    accounts
    .groupby("customer_id", as_index=False)
    .agg(
        account_count=("account_id", "nunique"),
        total_balance=("balance", "sum")
    )
    .query("account_count > 1")
    .sort_values(
        ["account_count", "total_balance"],
        ascending=[False, False]
    )
)

print("Q8 — Customers With Multiple Accounts")
display(multiple_accounts)


# Q9 — Which Customers Hold Multiple Accounts and Have High Balances?

multiple_high_value = (
    multiple_accounts
    .query("total_balance >= total_balance.quantile(0.75)")
    .sort_values("total_balance", ascending=False)
)

print("Q9 — Multiple-Account High-Value Customers")
display(multiple_high_value)


# Q10 — Which Customers Have Accounts + Loans + Cards?

account_customers = accounts["customer_id"].unique()

loan_customers = loan["customer_id"].unique()

card_customers = card["customer_id"].unique()
multi_product_customers = customers[
    customers["customer_id"].isin(account_customers)
    & customers["customer_id"].isin(loan_customers)
    & customers["customer_id"].isin(card_customers)
][
    [
        "customer_id",
        "name",
        "occupation",
        "annual_income",
        "credit_score",
        "city",
        "state"
    ]
]

print("Q10 — Customers With Accounts + Loans + Cards")
display(multi_product_customers)



Q1 — Occupation-wise Average Annual Income


,occupation,average_income,customer_count
4,Salaried - Private,1.834832e+06,8643
5,Self Employed,1.834169e+06,8492
6,Student,1.826686e+06,8428
0,Business Owner,1.820877e+06,8643
2,Retired,1.816193e+06,8509
3,Salaried - Government,1.812476e+06,8634
1,Freelancer,1.809176e+06,8651



Q2 — Occupation-wise Average Credit Score


,occupation,average_credit_score,customer_count
0,Business Owner,603.202360,8643
6,Student,603.049715,8428
3,Salaried - Government,600.905027,8634
5,Self Employed,600.879534,8492
2,Retired,599.162534,8509
4,Salaried - Private,596.673377,8643
1,Freelancer,594.147151,8651


Q3 — Customer Distribution by Credit Score Bands


...

,customer_id,account_count,total_balance
9689,12265,9,495653.35
34165,42991,8,557406.23
41682,52468,8,523103.31
9421,11922,8,501154.22
17542,22114,8,492137.72
...,...,...,...
27262,34358,2,2371.36
38716,48729,2,2087.62
5381,6782,2,1995.85
37591,47309,2,1918.86


Q9 — Multiple-Account High-Value Customers


,customer_id,account_count,total_balance
28963,36467,5,709888.78
44711,56318,4,689958.70
29830,37562,7,681456.63
33570,42253,5,669339.86
8243,10410,3,662374.12
...,...,...,...
47620,59972,2,164715.89
653,810,5,164706.58
36581,46059,5,164700.79
46700,58828,4,164691.30


Q10 — Customers With Accounts + Loans + Cards


,customer_id,name,occupation,annual_income,credit_score,city,state
0,1,Pooja Garcia,Retired,200025,690,Ahmedabad,Gujarat
1,2,Anjali Kapoor,Salaried - Government,1865003,672,Lucknow,Uttar Pradesh
6,7,Patricia Malhotra,Self Employed,2731327,886,Bengaluru,Karnataka
17,18,Kavya Jones,Salaried - Government,2444392,691,Pune,Maharashtra
18,19,Pooja Jones,Student,175931,396,Hyderabad,Telangana
...,...,...,...,...,...,...,...
59961,59962,Charles Gupta,Self Employed,3099276,631,Chandigarh,Punjab
59980,59981,William Miller,Student,508625,456,Indore,Madhya Pradesh
59987,59988,William Kumar,Self Employed,3272887,707,Ahmedabad,Gujarat
59988,59989,Amit Kumar,Freelancer,1267799,535,Kochi,Kerala


In [ ]:

# ============================================================
# GROUP 2 — BRANCH PERFORMANCE
# ============================================================


# Q1 — Which branches have the highest total deposits?

branch_deposits = (
    accounts
    .groupby("branch_id", as_index=False)
    .agg(
        total_deposits=("balance", "sum"),
        account_count=("account_id", "nunique")
    )
    .sort_values("total_deposits", ascending=False)
)

branch_deposits = branch_deposits.merge(
    branches[["branch_id", "branch_name", "city", "state"]],
    on="branch_id",
    how="left"
)

print("Q1 — Top Branches by Total Deposits")
display(branch_deposits.head(10))


# Q2 — Which branches have the highest number of customers?

branch_customers = (
    accounts
    .groupby("branch_id", as_index=False)
    .agg(
        customer_count=("customer_id", "nunique")
    )
    .sort_values("customer_count", ascending=False)
)

branch_customers = branch_customers.merge(
    branches[["branch_id", "branch_name", "city", "state"]],
    on="branch_id",
    how="left"
)

print("\nQ2 — Top Branches by Customer Count")
display(branch_customers.head(10))


# Q3 — Which branches issue the most loans?

branch_loans = (
    loan
    .groupby("branch_id", as_index=False)
    .agg(
        loan_count=("loan_id", "nunique"),
        total_loan_amount=("loan_amount", "sum")
    )
    .sort_values("loan_count", ascending=False)
)

branch_loans = branch_loans.merge(
    branches[["branch_id", "branch_name", "city", "state"]],
    on="branch_id",
    how="left"
)

print("\nQ3 — Top Branches by Loan Issuance")
display(branch_loans.head(10))


# Q4 — Which branches generate the highest transaction volume?

branch_transactions = (
    tranc
    .merge(
        accounts[["account_id", "branch_id"]],
        on="account_id",
        how="left"
    )
    .groupby("branch_id", as_index=False)
    .agg(
        transaction_count=("transaction_id", "nunique"),
        transaction_value=("amount", "sum")
    )
    .sort_values("transaction_count", ascending=False)
)

branch_transactions = branch_transactions.merge(
    branches[["branch_id", "branch_name", "city", "state"]],
    on="branch_id",
    how="left"
)

print("Q4 — Top Branches by Transaction Volume")
display(branch_transactions.head(10))



# Q5 — Which branches have the highest employee count?

branch_employees = (
    employee
    .groupby("branch_id", as_index=False)
    .agg(
        employee_count=("employee_id", "nunique"),
        average_salary=("salary", "mean")
    )
    .sort_values("employee_count", ascending=False)
)

branch_employees = branch_employees.merge(
    branches[["branch_id", "branch_name", "city", "state"]],
    on="branch_id",
    how="left"
)

print("\nQ5 — Top Branches by Employee Count")
display(branch_employees.head(10))


# Q6 — Which branches have high deposits but low loan activity?

branch_deposit_loan = (
    branch_deposits[[
        "branch_id",
        "total_deposits"
    ]]
    .merge(
        branch_loans[[
            "branch_id",
            "loan_count",
            "total_loan_amount"
        ]],
        on="branch_id",
        how="left"
    )
    .fillna(0)
)

deposit_threshold = branch_deposit_loan["total_deposits"].quantile(0.75)
loan_threshold = branch_deposit_loan["loan_count"].quantile(0.25)

high_deposit_low_loan = (
    branch_deposit_loan[
        (branch_deposit_loan["total_deposits"] >= deposit_threshold) &
        (branch_deposit_loan["loan_count"] <= loan_threshold)
    ]
    .sort_values("total_deposits", ascending=False)
)

high_deposit_low_loan = high_deposit_low_loan.merge(
    branches[["branch_id", "branch_name", "city", "state"]],
    on="branch_id",
    how="left"
)

print("\nQ6 — High Deposit but Low Loan Activity Branches")
display(high_deposit_low_loan)



# Q7 — Which branches perform strongly across -  deposits + customers + loans + transactions?

branch_performance = (
    branch_deposits[[
        "branch_id",
        "total_deposits"
    ]]
    .merge(
        branch_customers[[
            "branch_id",
            "customer_count"
        ]],
        on="branch_id",
        how="left"
    )
    .merge(
        branch_loans[[
            "branch_id",
            "loan_count"
        ]],
        on="branch_id",
        how="left"
    )
    .merge(
        branch_transactions[[
            "branch_id",
            "transaction_count"
        ]],
        on="branch_id",
        how="left"
    )
    .fillna(0)
)


branch_performance["deposit_rank"] = (
    branch_performance["total_deposits"]
    .rank(ascending=False, method="min")
)

branch_performance["customer_rank"] = (
    branch_performance["customer_count"]
    .rank(ascending=False, method="min")
)

branch_performance["loan_rank"] = (
    branch_performance["loan_count"]
    .rank(ascending=False, method="min")
)

branch_performance["transaction_rank"] = (
    branch_performance["transaction_count"]
    .rank(ascending=False, method="min")
)


branch_performance["average_rank"] = (
    branch_performance[
        [
            "deposit_rank",
            "customer_rank",
            "loan_rank",
            "transaction_rank"
        ]
    ].mean(axis=1)
)


branch_performance = (
    branch_performance
    .sort_values("average_rank")
)

branch_performance = branch_performance.merge(
    branches[[
        "branch_id",
        "branch_name",
        "city",
        "state"
    ]],
    on="branch_id",
    how="left"
)

print("Q7 — Overall Branch Performance Ranking")
display(branch_performance.head(10))



Q1 — Top Branches by Total Deposits


,branch_id,total_deposits,account_count,branch_name,city,state
0,43,32619069.04,668,Chandigarh Branch 7,Chandigarh,Punjab
1,8,32396876.95,669,Chandigarh Branch 8,Chandigarh,Punjab
2,113,32300797.90,662,Jaipur Branch 5,Jaipur,Rajasthan
3,70,32081996.33,655,Ahmedabad Branch 7,Ahmedabad,Gujarat
4,12,32077846.13,667,Indore Branch 3,Indore,Madhya Pradesh
5,29,32047429.89,670,Jaipur Branch 2,Jaipur,Rajasthan
6,79,31978103.66,659,Jaipur Branch 7,Jaipur,Rajasthan
7,58,31969232.54,700,Delhi Branch 4,Delhi,Delhi
8,150,31889032.84,675,Mumbai Branch 6,Mumbai,Maharashtra
9,92,31870099.08,645,Kochi Branch 2,Kochi,Kerala



Q2 — Top Branches by Customer Count


,branch_id,customer_count,branch_name,city,state
0,132,699,Jaipur Branch 6,Jaipur,Rajasthan
1,58,696,Delhi Branch 4,Delhi,Delhi
2,15,681,Chandigarh Branch 6,Chandigarh,Punjab
3,150,672,Mumbai Branch 6,Mumbai,Maharashtra
4,66,671,Chennai Branch 3,Chennai,Tamil Nadu
5,77,669,Chennai Branch 5,Chennai,Tamil Nadu
6,8,667,Chandigarh Branch 8,Chandigarh,Punjab
7,50,666,Bhopal Branch 5,Bhopal,Madhya Pradesh
8,29,666,Jaipur Branch 2,Jaipur,Rajasthan
9,12,665,Indore Branch 3,Indore,Madhya Pradesh



Q3 — Top Branches by Loan Issuance


...

,branch_id,employee_count,average_salary,branch_name,city,state
0,149,22,108655.090909,Kochi Branch 5,Kochi,Kerala
1,53,21,81406.428571,Hyderabad Branch 8,Hyderabad,Telangana
2,114,20,94631.700000,Kolkata Branch 6,Kolkata,West Bengal
3,51,19,118990.368421,Bhopal Branch 6,Bhopal,Madhya Pradesh
4,40,19,94725.368421,Lucknow Branch 4,Lucknow,Uttar Pradesh
5,140,19,98994.947368,Kolkata Branch 5,Kolkata,West Bengal
6,79,19,102910.157895,Jaipur Branch 7,Jaipur,Rajasthan
7,57,18,90816.000000,Jaipur Branch 3,Jaipur,Rajasthan
8,39,18,104721.555556,Chennai Branch 3,Chennai,Tamil Nadu
9,11,18,97474.944444,Ahmedabad Branch 2,Ahmedabad,Gujarat



Q6 — High Deposit but Low Loan Activity Branches


,branch_id,total_deposits,loan_count,total_loan_amount,branch_name,city,state
0,12,32077846.13,137,61660029.65,Indore Branch 3,Indore,Madhya Pradesh
1,79,31978103.66,129,45320125.60,Jaipur Branch 7,Jaipur,Rajasthan
2,58,31969232.54,135,45707624.99,Delhi Branch 4,Delhi,Delhi
3,92,31870099.08,137,68014603.83,Kochi Branch 2,Kochi,Kerala
4,103,31815632.48,138,60438164.64,Kochi Branch 4,Kochi,Kerala
5,141,31645123.39,138,58697556.99,Indore Branch 6,Indore,Madhya Pradesh
6,104,31491647.71,111,50970216.93,Mumbai Branch 5,Mumbai,Maharashtra
7,15,31435234.17,137,62223709.01,Chandigarh Branch 6,Chandigarh,Punjab
8,111,30927445.52,136,55206357.56,Chennai Branch 3,Chennai,Tamil Nadu
9,94,30766256.11,133,59458787.64,Hyderabad Branch 4,Hyderabad,Telangana


Q7 — Overall Branch Performance Ranking


,branch_id,total_deposits,customer_count,loan_count,transaction_count,deposit_rank,customer_rank,loan_rank,transaction_rank,average_rank,branch_name,city,state
0,66,30771715.18,671,168,14229,23.0,5.0,8.0,5.0,10.25,Chennai Branch 3,Chennai,Tamil Nadu
1,29,32047429.89,666,160,14100,6.0,8.0,25.0,10.0,12.25,Jaipur Branch 2,Jaipur,Rajasthan
2,43,32619069.04,662,159,14082,1.0,14.0,28.0,12.0,13.75,Chandigarh Branch 7,Chandigarh,Punjab
3,132,31837894.48,699,150,14672,11.0,1.0,57.0,2.0,17.75,Jaipur Branch 6,Jaipur,Rajasthan
4,70,32081996.33,651,161,13875,4.0,27.0,20.0,21.0,18.00,Ahmedabad Branch 7,Ahmedabad,Gujarat
5,150,31889032.84,672,150,14261,9.0,4.0,57.0,4.0,18.50,Mumbai Branch 6,Mumbai,Maharashtra
6,77,30937762.89,669,152,14198,21.0,6.0,43.0,6.0,19.00,Chennai Branch 5,Chennai,Tamil Nadu
7,100,31681188.63,651,171,13753,14.0,27.0,5.0,30.0,19.00,Delhi Branch 1,Delhi,Delhi
8,113,32300797.90,655,152,13992,3.0,19.0,43.0,15.0,20.00,Jaipur Branch 5,Jaipur,Rajasthan
9,101,29929083.11,655,179,13673,34.0,19.0,2.0,37.0,23.00,Kochi Branch 2,Kochi,Kerala


In [42]:
# ============================================================
# GROUP 3 — ACCOUNT & DEPOSIT ANALYSIS
# ============================================================


# Q1 — Which account types are most common?

account_type_count = (
    accounts
    .groupby("account_type", as_index=False)
    .agg(
        account_count=("account_id", "nunique")
    )
    .sort_values("account_count", ascending=False)
)

print("Q1 — Most Common Account Types")
display(account_type_count)


# Q2 — Which account types hold the most total balance?

account_type_balance = (
    accounts
    .groupby("account_type", as_index=False)
    .agg(
        total_balance=("balance", "sum"),
        account_count=("account_id", "nunique")
    )
    .sort_values("total_balance", ascending=False)
)

print("\nQ2 — Account Types by Total Balance")
display(account_type_balance)


# Q3 — Which account types have the highest average balance?

account_type_avg_balance = (
    accounts
    .groupby("account_type", as_index=False)
    .agg(
        average_balance=("balance", "mean"),
        account_count=("account_id", "nunique")
    )
    .sort_values("average_balance", ascending=False)
)

print("\nQ3 — Account Types by Average Balance")
display(account_type_avg_balance)


# Q4 — Which branches hold the highest total deposits?

branch_deposits = (
    accounts
    .groupby("branch_id", as_index=False)
    .agg(
        total_deposits=("balance", "sum"),
        account_count=("account_id", "nunique")
    )
    .sort_values("total_deposits", ascending=False)
)

branch_deposits = branch_deposits.merge(
    branches[["branch_id", "branch_name", "city", "state"]],
    on="branch_id",
    how="left"
)

print("\nQ4 — Branches with Highest Total Deposits")
display(branch_deposits.head(10))


# Q5 — Which branches have high customer count
#       but relatively low deposits?

branch_customer_deposits = (
    accounts
    .groupby("branch_id", as_index=False)
    .agg(
        customer_count=("customer_id", "nunique"),
        total_deposits=("balance", "sum")
    )
)

# Define high customer count and low deposits
customer_threshold = (
    branch_customer_deposits["customer_count"]
    .quantile(0.75)
)

deposit_threshold = (
    branch_customer_deposits["total_deposits"]
    .quantile(0.25)
)

high_customers_low_deposits = (
    branch_customer_deposits[
        (branch_customer_deposits["customer_count"] >= customer_threshold) &
        (branch_customer_deposits["total_deposits"] <= deposit_threshold)
    ]
    .sort_values("customer_count", ascending=False)
)

high_customers_low_deposits = high_customers_low_deposits.merge(
    branches[["branch_id", "branch_name", "city", "state"]],
    on="branch_id",
    how="left"
)

print("\nQ5 — High Customer Count but Relatively Low Deposits")
display(high_customers_low_deposits)


# Q6 — What percentage of customers own multiple accounts?

customer_account_count = (
    accounts
    .groupby("customer_id", as_index=False)
    .agg(
        account_count=("account_id", "nunique")
    )
)

multiple_account_customers = (
    customer_account_count["account_count"] > 1
).sum()

total_customers = customers["customer_id"].nunique()

multiple_account_percentage = (
    multiple_account_customers
    / total_customers
    * 100
)

print("\nQ6 — Percentage of Customers with Multiple Accounts")
print(f"Customers with multiple accounts: {multiple_account_customers:,}")
print(f"Total customers: {total_customers:,}")
print(f"Percentage: {multiple_account_percentage:.2f}%")


# Q7 — Which customers/branches have unusually high account balances?

# Customer-level total balance
customer_high_balance = (
    accounts
    .groupby("customer_id", as_index=False)
    .agg(
        total_balance=("balance", "sum"),
        account_count=("account_id", "nunique")
    )
)

# 99th percentile = unusually high balance
customer_balance_threshold = (
    customer_high_balance["total_balance"]
    .quantile(0.99)
)

high_balance_customers = (
    customer_high_balance[
        customer_high_balance["total_balance"] >= customer_balance_threshold
    ]
    .sort_values("total_balance", ascending=False)
)

high_balance_customers = high_balance_customers.merge(
    customers[
        [
            "customer_id",
            "name",
            "occupation",
            "city",
            "state"
        ]
    ],
    on="customer_id",
    how="left"
)

print("\nQ7A — Customers with Unusually High Account Balances")
display(high_balance_customers.head(20))


# Branch-level total balance
branch_high_balance = (
    accounts
    .groupby("branch_id", as_index=False)
    .agg(
        total_balance=("balance", "sum"),
        account_count=("account_id", "nunique")
    )
)

branch_balance_threshold = (
    branch_high_balance["total_balance"]
    .quantile(0.99)
)

high_balance_branches = (
    branch_high_balance[
        branch_high_balance["total_balance"] >= branch_balance_threshold
    ]
    .sort_values("total_balance", ascending=False)
)

high_balance_branches = high_balance_branches.merge(
    branches[
        [
            "branch_id",
            "branch_name",
            "city",
            "state"
        ]
    ],
    on="branch_id",
    how="left"
)

print("\nQ7B — Branches with Unusually High Account Balances")
display(high_balance_branches)

Q1 — Most Common Account Types


,account_type,account_count
4,Savings,47557
0,Current,19021
3,Salary,14270
1,Fixed Deposit,9458
2,NRI,4694



Q2 — Account Types by Total Balance


,account_type,total_balance,account_count
4,Savings,2.187487e+09,47557
0,Current,8.701127e+08,19021
3,Salary,6.449175e+08,14270
1,Fixed Deposit,4.257899e+08,9458
2,NRI,2.190108e+08,4694



Q3 — Account Types by Average Balance


...

,branch_id,customer_count,total_deposits,branch_name,city,state
0,136,660,27833902.47,Chennai Branch 1,Chennai,Tamil Nadu



Q6 — Percentage of Customers with Multiple Accounts
Customers with multiple accounts: 28,172
Total customers: 60,000
Percentage: 46.95%

Q7A — Customers with Unusually High Account Balances


,customer_id,total_balance,account_count,name,occupation,city,state
0,36467,709888.78,5,Ajay Miller,Freelancer,Ahmedabad,Gujarat
1,56318,689958.70,4,Elizabeth Brown,Salaried - Private,Mumbai,Maharashtra
2,37562,681456.63,7,William Kapoor,Salaried - Private,Kochi,Kerala
3,42253,669339.86,5,Elizabeth Smith,Retired,Bhopal,Madhya Pradesh
4,10410,662374.12,3,Ajay Martinez,Business Owner,Mumbai,Maharashtra
5,7693,658523.34,3,Sneha Singh,Freelancer,Jaipur,Rajasthan
6,50386,648350.33,7,Karen Johnson,Self Employed,Pune,Maharashtra
7,13958,644399.02,6,Neha Reddy,Salaried - Private,Bengaluru,Karnataka
8,24476,641481.00,5,Pooja Davis,Freelancer,Pune,Maharashtra
9,53726,634851.44,2,Vikram Nair,Salaried - Government,Pune,Maharashtra



Q7B — Branches with Unusually High Account Balances


,branch_id,total_balance,account_count,branch_name,city,state
0,43,32619069.04,668,Chandigarh Branch 7,Chandigarh,Punjab
1,8,32396876.95,669,Chandigarh Branch 8,Chandigarh,Punjab


In [ ]:
# ============================================================
# GROUP 4 — LOAN & CREDIT RISK
# ============================================================


# Q1 — What is the loan portfolio distribution by loan type?

loan_type_distribution = (
    loan
    .groupby("loan_type", as_index=False)
    .agg(
        loan_count=("loan_id", "nunique"),
        total_loan_amount=("loan_amount", "sum"),
        average_loan_amount=("loan_amount", "mean")
    )
    .sort_values("total_loan_amount", ascending=False)
)

loan_type_distribution["portfolio_percentage"] = (
    loan_type_distribution["total_loan_amount"]
    / loan_type_distribution["total_loan_amount"].sum()
    * 100
)

print("Q1 — Loan Portfolio Distribution by Loan Type")
display(loan_type_distribution)


# Q2 — Which loan types have the highest average loan amount?

loan_type_average = (
    loan
    .groupby("loan_type", as_index=False)
    .agg(
        average_loan_amount=("loan_amount", "mean"),
        loan_count=("loan_id", "nunique")
    )
    .sort_values("average_loan_amount", ascending=False)
)

print("\nQ2 — Loan Types by Average Loan Amount")
display(loan_type_average)


# Q3 — What percentage of loans are
#       Active / Closed / Defaulted / Written Off?

loan_status_analysis = (
    loan
    .groupby("status", as_index=False)
    .agg(
        loan_count=("loan_id", "nunique"),
        total_loan_amount=("loan_amount", "sum")
    )
)

loan_status_analysis["percentage"] = (
    loan_status_analysis["loan_count"]
    / loan_status_analysis["loan_count"].sum()
    * 100
)

loan_status_analysis = loan_status_analysis.sort_values(
    "percentage",
    ascending=False
)

print("\nQ3 — Loan Status Distribution")
display(loan_status_analysis)


# Q4 — Which loan types have the highest default rate?

loan_default_rate = (
    loan
    .groupby("loan_type", as_index=False)
    .agg(
        total_loans=("loan_id", "nunique"),
        defaulted_loans=("status", lambda x: (x == "Defaulted").sum())
    )
)

loan_default_rate["default_rate"] = (
    loan_default_rate["defaulted_loans"]
    / loan_default_rate["total_loans"]
    * 100
)

loan_default_rate = loan_default_rate.sort_values(
    "default_rate",
    ascending=False
)

print("\nQ4 — Loan Types with Highest Default Rate")
display(loan_default_rate)


# Q5 — Which branches have the highest loan default rate?

branch_default_rate = (
    loan
    .groupby("branch_id", as_index=False)
    .agg(
        total_loans=("loan_id", "nunique"),
        defaulted_loans=("status", lambda x: (x == "Defaulted").sum())
    )
)

branch_default_rate["default_rate"] = (
    branch_default_rate["defaulted_loans"]
    / branch_default_rate["total_loans"]
    * 100
)

branch_default_rate = (
    branch_default_rate
    .sort_values("default_rate", ascending=False)
    .merge(
        branches[["branch_id", "branch_name", "city", "state"]],
        on="branch_id",
        how="left"
    )
)

print("\nQ5 — Branches with Highest Loan Default Rate")
display(branch_default_rate.head(10))


# Q6 — Which customers have repeated late payments?

customer_late_payments = (
    loan_pay
    .merge(
        loan[["loan_id", "customer_id"]],
        on="loan_id",
        how="left"
    )
    .groupby("customer_id", as_index=False)
    .agg(
        late_payment_count=("late_payment_flag", "sum"),
        total_payments=("payment_id", "nunique")
    )
)

repeated_late_payers = (
    customer_late_payments[
        customer_late_payments["late_payment_count"] > 1
    ]
    .sort_values("late_payment_count", ascending=False)
)

repeated_late_payers = repeated_late_payers.merge(
    customers[
        [
            "customer_id",
            "name",
            "occupation",
            "credit_score",
            "annual_income"
        ]
    ],
    on="customer_id",
    how="left"
)

print("\nQ6 — Customers with Repeated Late Payments")
display(repeated_late_payers.head(20))


# Q7 — Which loan types have the highest late-payment rate?

loan_payment_late = (
    loan_pay
    .merge(
        loan[["loan_id", "loan_type"]],
        on="loan_id",
        how="left"
    )
    .groupby("loan_type", as_index=False)
    .agg(
        total_payments=("payment_id", "nunique"),
        late_payments=("late_payment_flag", "sum")
    )
)

loan_payment_late["late_payment_rate"] = (
    loan_payment_late["late_payments"]
    / loan_payment_late["total_payments"]
    * 100
)

loan_payment_late = loan_payment_late.sort_values(
    "late_payment_rate",
    ascending=False
)

print("\nQ7 — Loan Types with Highest Late-Payment Rate")
display(loan_payment_late)


# Q8 — How much principal/interest has been paid vs outstanding?

total_loan_amount = loan["loan_amount"].sum()

principal_paid = loan_pay["principal_component"].sum()
interest_paid = loan_pay["interest_component"].sum()
total_paid = loan_pay["amount_paid"].sum()

principal_outstanding = total_loan_amount - principal_paid

payment_summary = pd.DataFrame({
    "metric": [
        "Total Loan Principal",
        "Principal Paid",
        "Principal Outstanding",
        "Interest Paid",
        "Total Amount Paid"
    ],
    "amount": [
        total_loan_amount,
        principal_paid,
        principal_outstanding,
        interest_paid,
        total_paid
    ]
})

print("Q8 — Principal / Interest Paid vs Outstanding")
display(payment_summary)

# Q9 — Which customers combine low credit score
#       + high loan exposure + late payments?


customer_loan_exposure = (
    loan
    .groupby("customer_id", as_index=False)
    .agg(
        total_loan_exposure=("loan_amount", "sum"),
        loan_count=("loan_id", "nunique")
    )
)


customer_late_payments = (
    loan_pay
    .merge(
        loan[["loan_id", "customer_id"]],
        on="loan_id",
        how="left"
    )
    .groupby("customer_id", as_index=False)
    .agg(
        late_payment_count=("late_payment_flag", "sum")
    )
)


customer_risk = (
    customers[
        [
            "customer_id",
            "name",
            "credit_score",
            "annual_income",
            "occupation"
        ]
    ]
    .merge(
        customer_loan_exposure,
        on="customer_id",
        how="inner"
    )
    .merge(
        customer_late_payments,
        on="customer_id",
        how="left"
    )
    .fillna({"late_payment_count": 0})
)

# Step 4 — Define high loan exposure
loan_exposure_threshold = (
    customer_risk["total_loan_exposure"]
    .quantile(0.75)
)


high_risk_customers = (
    customer_risk[
        (customer_risk["credit_score"] < 600) &
        (customer_risk["total_loan_exposure"] >= loan_exposure_threshold) &
        (customer_risk["late_payment_count"] > 0)
    ]
    .sort_values(
        ["late_payment_count", "total_loan_exposure"],
        ascending=[False, False]
    )
)

print("Q9 — Low Credit + High Loan Exposure + Late Payments")
display(high_risk_customers.head(20))

Q1 — Loan Portfolio Distribution by Loan Type


,loan_type,loan_count,total_loan_amount,average_loan_amount,portfolio_percentage
4,Home Loan,3659,1.548369e+09,423167.225947,16.803343
5,Personal Loan,3632,1.546647e+09,425839.060022,16.784662
1,Business Loan,3691,1.539605e+09,417124.155662,16.708238
3,Gold Loan,3685,1.533235e+09,416074.659788,16.639107
0,Auto Loan,3631,1.526908e+09,420520.091644,16.570449
2,Education Loan,3702,1.519882e+09,410557.130581,16.494201



Q2 — Loan Types by Average Loan Amount


,loan_type,average_loan_amount,loan_count
5,Personal Loan,425839.060022,3632
4,Home Loan,423167.225947,3659
0,Auto Loan,420520.091644,3631
1,Business Loan,417124.155662,3691
3,Gold Loan,416074.659788,3685
2,Education Loan,410557.130581,3702



Q3 — Loan Status Distribution


...

,loan_type,total_payments,late_payments,late_payment_rate
0,Auto Loan,99013,12067,12.187289
4,Home Loan,100056,12096,12.089230
2,Education Loan,101448,12242,12.067266
5,Personal Loan,98627,11812,11.976436
3,Gold Loan,100571,11939,11.871215
1,Business Loan,100285,11882,11.848233


Q8 — Principal / Interest Paid vs Outstanding


,metric,amount
0,Total Loan Principal,9.214648e+09
1,Principal Paid,5.092382e+09
2,Principal Outstanding,4.122266e+09
3,Interest Paid,1.146094e+09
4,Total Amount Paid,6.238476e+09


Q9 — Low Credit + High Loan Exposure + Late Payments


,customer_id,name,credit_score,annual_income,occupation,total_loan_exposure,loan_count,late_payment_count
10306,33771,Anita Verma,394,692293,Student,1500646.94,3,25
2120,6811,Barbara Pillai,381,2023552,Business Owner,1322506.93,4,23
9579,31336,Manoj Davis,523,889613,Student,2992904.74,7,21
3691,11929,Thomas Mehta,560,216168,Retired,2074706.82,3,18
18248,59572,John Chopra,484,2296338,Freelancer,1070350.25,3,18
3545,11479,Karen Mehta,468,2894276,Salaried - Government,1941126.52,3,17
5793,19002,William Chopra,368,520621,Self Employed,1719565.38,3,17
10117,33115,Amit Bose,528,567310,Freelancer,1070880.21,3,17
18381,59995,Sneha Kapoor,483,2907694,Student,3088791.89,4,16
13212,43154,Michael Williams,307,759446,Business Owner,3037847.84,4,16


In [44]:
# ============================================================
# GROUP 5 — CARDS & FRAUD
# ============================================================


# Q1 — What is the distribution of cards by card type?

card_type_distribution = (
    card
    .groupby("card_type", as_index=False)
    .agg(
        card_count=("card_id", "nunique")
    )
    .sort_values("card_count", ascending=False)
)

card_type_distribution["percentage"] = (
    card_type_distribution["card_count"]
    / card_type_distribution["card_count"].sum()
    * 100
)

print("Q1 — Card Distribution by Card Type")
display(card_type_distribution)


# Q2 — Which card types have the highest credit limits?

card_type_credit_limit = (
    card
    .groupby("card_type", as_index=False)
    .agg(
        average_credit_limit=("credit_limit", "mean"),
        total_credit_limit=("credit_limit", "sum"),
        card_count=("card_id", "nunique")
    )
    .sort_values("average_credit_limit", ascending=False)
)

print("\nQ2 — Card Types by Credit Limit")
display(card_type_credit_limit)


# Q3 — Which merchant categories generate the highest card spending?

merchant_spending = (
    card_trans
    .groupby("merchant_category", as_index=False)
    .agg(
        transaction_count=("card_txn_id", "nunique"),
        total_spending=("amount", "sum"),
        average_transaction=("amount", "mean")
    )
    .sort_values("total_spending", ascending=False)
)

print("\nQ3 — Merchant Categories by Card Spending")
display(merchant_spending)


# Q4 — Which merchant categories have the highest fraud rate?

merchant_fraud = (
    card_trans
    .groupby("merchant_category", as_index=False)
    .agg(
        transaction_count=("card_txn_id", "nunique"),
        fraudulent_transactions=("is_fraud", "sum")
    )
)

merchant_fraud["fraud_rate"] = (
    merchant_fraud["fraudulent_transactions"]
    / merchant_fraud["transaction_count"]
    * 100
)

merchant_fraud = merchant_fraud.sort_values(
    "fraud_rate",
    ascending=False
)

print("\nQ4 — Merchant Categories with Highest Fraud Rate")
display(merchant_fraud)


# Q5 — Which card types have the highest fraud rate?

card_type_fraud = (
    card_trans
    .merge(
        card[["card_id", "card_type"]],
        on="card_id",
        how="left"
    )
    .groupby("card_type", as_index=False)
    .agg(
        transaction_count=("card_txn_id", "nunique"),
        fraudulent_transactions=("is_fraud", "sum")
    )
)

card_type_fraud["fraud_rate"] = (
    card_type_fraud["fraudulent_transactions"]
    / card_type_fraud["transaction_count"]
    * 100
)

card_type_fraud = card_type_fraud.sort_values(
    "fraud_rate",
    ascending=False
)

print("\nQ5 — Card Types with Highest Fraud Rate")
display(card_type_fraud)


# Q6 — What is the average transaction amount for
#       fraudulent vs legitimate transactions?

fraud_transaction_analysis = (
    card_trans
    .assign(
        transaction_status=card_trans["is_fraud"].map({
            0: "Legitimate",
            1: "Fraudulent"
        })
    )
    .groupby("transaction_status", as_index=False)
    .agg(
        transaction_count=("card_txn_id", "nunique"),
        total_amount=("amount", "sum"),
        average_transaction_amount=("amount", "mean")
    )
)

print("\nQ6 — Fraudulent vs Legitimate Transaction Amount")
display(fraud_transaction_analysis)

Q1 — Card Distribution by Card Type


,card_type,card_count,percentage
3,Debit,35700,54.923077
0,Credit - Classic,16300,25.076923
1,Credit - Gold,8367,12.872308
2,Credit - Platinum,4633,7.127692



Q2 — Card Types by Credit Limit


,card_type,average_credit_limit,total_credit_limit,card_count
2,Credit - Platinum,42262.033240,195800000,4633
3,Debit,39658.263305,1415800000,35700
0,Credit - Classic,39397.239264,642175000,16300
1,Credit - Gold,38338.114019,320775000,8367



Q3 — Merchant Categories by Card Spending


...

,merchant_category,transaction_count,fraudulent_transactions,fraud_rate
0,ATM Withdrawal,214289,1121,0.523125
7,Healthcare,214975,1119,0.520526
10,Salary Credit,214558,1115,0.519673
3,Entertainment,213605,1096,0.513097
4,Fuel,214392,1076,0.501884
13,Utilities,214654,1069,0.498011
6,Groceries,214009,1065,0.497643
5,Fund Transfer,214964,1067,0.496362
1,Dining,214599,1057,0.492547
9,Rent,213915,1046,0.488979



Q5 — Card Types with Highest Fraud Rate


,card_type,transaction_count,fraudulent_transactions,fraud_rate
0,Credit - Classic,752873,3818,0.507124
1,Credit - Gold,386252,1927,0.498897
3,Debit,1647839,8193,0.497197
2,Credit - Platinum,213036,1016,0.476915



Q6 — Fraudulent vs Legitimate Transaction Amount


,transaction_status,transaction_count,total_amount,average_transaction_amount
0,Fraudulent,14954,2.741814e+07,1833.498946
1,Legitimate,2985046,5.523686e+09,1850.452425


In [45]:
# ============================================================
# GROUP 6 — TRANSACTION ANALYSIS
# ============================================================


# Q1 — Which transaction types generate the highest
#       transaction value?

transaction_type_analysis = (
    tranc
    .groupby("txn_type", as_index=False)
    .agg(
        transaction_count=("transaction_id", "nunique"),
        total_transaction_value=("amount", "sum"),
        average_transaction_value=("amount", "mean")
    )
    .sort_values("total_transaction_value", ascending=False)
)

print("Q1 — Transaction Types by Highest Transaction Value")
display(transaction_type_analysis)


# Q2 — Which channels are used most frequently?

channel_analysis = (
    tranc
    .groupby("channel", as_index=False)
    .agg(
        transaction_count=("transaction_id", "nunique"),
        total_transaction_value=("amount", "sum")
    )
    .sort_values("transaction_count", ascending=False)
)

channel_analysis["percentage"] = (
    channel_analysis["transaction_count"]
    / channel_analysis["transaction_count"].sum()
    * 100
)

print("\nQ2 — Most Frequently Used Transaction Channels")
display(channel_analysis)


# Q3 — What are the monthly transaction volume
#       and transaction-value trends?

monthly_transaction_analysis = (
    tranc
    .assign(
        month=tranc["txn_date"].dt.to_period("M")
    )
    .groupby("month", as_index=False)
    .agg(
        transaction_count=("transaction_id", "nunique"),
        total_transaction_value=("amount", "sum"),
        average_transaction_value=("amount", "mean")
    )
    .sort_values("month")
)

print("\nQ3 — Monthly Transaction Volume and Value Trends")
display(monthly_transaction_analysis)


# Q4 — How does deposit vs withdrawal activity
#       change over time?

deposit_withdrawal_analysis = (
    tranc[
        tranc["txn_type"].isin(["Deposit", "Withdrawal"])
    ]
    .assign(
        month=lambda x: x["txn_date"].dt.to_period("M")
    )
    .groupby(
        ["month", "txn_type"],
        as_index=False
    )
    .agg(
        transaction_count=("transaction_id", "nunique"),
        total_amount=("amount", "sum")
    )
    .sort_values(["month", "txn_type"])
)

print("\nQ4 — Monthly Deposit vs Withdrawal Activity")
display(deposit_withdrawal_analysis)


# Q5 — Which merchant categories have the highest
#       transaction value over time?

merchant_category_analysis = (
    tranc
    .assign(
        month=lambda x: x["txn_date"].dt.to_period("M")
    )
    .groupby(
        ["month", "merchant_category"],
        as_index=False
    )
    .agg(
        transaction_count=("transaction_id", "nunique"),
        total_transaction_value=("amount", "sum")
    )
    .sort_values(
        ["month", "total_transaction_value"],
        ascending=[True, False]
    )
)

print("\nQ5 — Merchant Categories with Highest Transaction Value Over Time")
display(merchant_category_analysis)

Q1 — Transaction Types by Highest Transaction Value


,txn_type,transaction_count,total_transaction_value,average_transaction_value
0,Deposit,498861,3.018981e+09,6051.747686
5,Withdrawal,499405,3.017945e+09,6043.080477
4,Transfer Out,301321,1.828299e+09,6067.610638
3,Transfer In,300258,1.816111e+09,6048.503104
1,Fee Debit,200209,1.208979e+09,6038.586661
2,Interest Credit,199946,1.206454e+09,6033.901047



Q2 — Most Frequently Used Transaction Channels


,channel,transaction_count,total_transaction_value,percentage
2,Mobile App,333776,2.018263e+09,16.68880
4,POS,333487,2.019633e+09,16.67435
0,ATM,333383,2.019315e+09,16.66915
1,Branch,333369,2.013472e+09,16.66845
5,UPI,333106,2.015659e+09,16.65530
3,Online Banking,332879,2.010427e+09,16.64395



Q3 — Monthly Transaction Volume and Value Trends


,month,transaction_count,total_transaction_value,average_transaction_value
0,2018-01,20003,1.197059e+08,5984.395949
1,2018-02,18032,1.097275e+08,6085.153769
2,2018-03,19921,1.210276e+08,6075.376280
3,2018-04,19196,1.155013e+08,6016.945005
4,2018-05,20162,1.225683e+08,6079.174125
...,...,...,...,...
97,2026-02,18011,1.094937e+08,6079.266750
98,2026-03,20035,1.227803e+08,6128.289390
99,2026-04,19380,1.169401e+08,6034.060038
100,2026-05,20150,1.228901e+08,6098.763797



Q4 — Monthly Deposit vs Withdrawal Activity


,month,txn_type,transaction_count,total_amount
0,2018-01,Deposit,4886,29732507.38
1,2018-01,Withdrawal,5029,29939114.55
2,2018-02,Deposit,4435,27114705.62
3,2018-02,Withdrawal,4422,26582969.53
4,2018-03,Deposit,4928,30557608.68
...,...,...,...,...
199,2026-04,Withdrawal,4817,29002368.76
200,2026-05,Deposit,5069,31203584.82
201,2026-05,Withdrawal,5047,30838906.55
202,2026-06,Deposit,4856,29693323.61



Q5 — Merchant Categories with Highest Transaction Value Over Time


,month,merchant_category,transaction_count,total_transaction_value
13,2018-01,Utilities,1493,9027007.65
0,2018-01,ATM Withdrawal,1498,8927440.22
9,2018-01,Rent,1418,8777494.77
6,2018-01,Groceries,1393,8697741.21
12,2018-01,Travel,1446,8694015.62
...,...,...,...,...
1426,2026-06,Travel,1347,8446222.63
1415,2026-06,Dining,1414,8333191.02
1420,2026-06,Groceries,1390,8332976.20
1423,2026-06,Rent,1381,8286571.54


In [32]:
import pandas as pd
import csv

sup_tic = pd.read_csv(
    "../all_csv/support_tickets-_1_.csv",
    sep=",",
    quoting=csv.QUOTE_NONE
)

# Clean column names
sup_tic.columns = sup_tic.columns.str.strip().str.strip('"')

# Clean string values
for col in sup_tic.select_dtypes(include="str").columns:
    sup_tic[col] = sup_tic[col].str.strip().str.strip('"')

# Convert numeric columns
for col in ["ticket_id", "customer_id", "satisfaction_score"]:
    if col in sup_tic.columns:
        sup_tic[col] = pd.to_numeric(sup_tic[col], errors="coerce")

# Convert dates
for col in ["date_opened", "date_resolved"]:
    if col in sup_tic.columns:
        sup_tic[col] = pd.to_datetime(sup_tic[col], errors="coerce")

print(sup_tic.shape)
print(sup_tic.columns.tolist())
customers = pd.read_csv(
    "../all_csv/customers.csv",
    sep=",",
    quoting=csv.QUOTE_NONE)

print(customers.columns)

(25000, 7)
['ticket_id', 'customer_id', 'issue_type', 'date_opened', 'date_resolved', 'status', 'satisfaction_score']
Index(['"customer_id', 'name', 'gender', 'date_of_birth', 'city', 'state',
       'phone', 'email', 'occupation', 'annual_income', 'join_date',
       'credit_score"'],
      dtype='str')


In [46]:
# ============================================================
# GROUP 7 — CUSTOMER SUPPORT & EXPERIENCE
# ============================================================


# Q1 — Which support issues occur most frequently?

support_issue_analysis = (
     sup_tic.groupby("issue_type", as_index=False)
    .agg(
        ticket_count=("ticket_id", "nunique")
    )
    .sort_values("ticket_count", ascending=False)
)

support_issue_analysis["percentage"] = (
    support_issue_analysis["ticket_count"]
    / support_issue_analysis["ticket_count"].sum()
    * 100
)

print("Q1 — Most Frequent Support Issues")
display(support_issue_analysis)


# Q2 — Which issues have the lowest satisfaction scores?

issue_satisfaction = (
    sup_tic
    .groupby("issue_type", as_index=False)
    .agg(
        average_satisfaction=("satisfaction_score", "mean"),
        ticket_count=("ticket_id", "nunique")
    )
    .sort_values("average_satisfaction", ascending=True)
)

print("\nQ2 — Issues with Lowest Satisfaction Scores")
display(issue_satisfaction)


# Q3 — Which support issues take longest to resolve?

# Calculate resolution time in days
support_resolution = sup_tic.copy()

support_resolution["resolution_days"] = (
    support_resolution["date_resolved"]
    - support_resolution["date_opened"]
).dt.total_seconds() / (60 * 60 * 24)

issue_resolution_analysis = (
    support_resolution
    .groupby("issue_type", as_index=False)
    .agg(
        average_resolution_days=("resolution_days", "mean"),
        ticket_count=("ticket_id", "nunique")
    )
    .sort_values("average_resolution_days", ascending=False)
)

print("\nQ3 — Support Issues with Longest Resolution Time")
display(issue_resolution_analysis)


# Q4 — Does resolution time affect satisfaction?

resolution_satisfaction = (
    support_resolution[
        support_resolution["resolution_days"].notna()
    ]
    .groupby("issue_type", as_index=False)
    .agg(
        average_resolution_days=("resolution_days", "mean"),
        average_satisfaction=("satisfaction_score", "mean"),
        ticket_count=("ticket_id", "nunique")
    )
    .sort_values("average_resolution_days", ascending=False)
)

resolution_satisfaction_correlation = (
    resolution_satisfaction["average_resolution_days"]
    .corr(resolution_satisfaction["average_satisfaction"])
)

print("\nQ4 — Resolution Time vs Satisfaction")
display(resolution_satisfaction)

print(
    f"Correlation between resolution time and satisfaction: "
    f"{resolution_satisfaction_correlation:.3f}"
)


Q1 — Most Frequent Support Issues


,issue_type,ticket_count,percentage
2,Card Blocked,2560,10.240
4,Fraud Report,2535,10.140
8,Net Banking Issue,2532,10.128
3,Cheque Bounce,2531,10.124
7,Loan Query,2521,10.084
6,KYC Update,2491,9.964
0,Account Statement,2474,9.896
9,Wrong Debit,2466,9.864
1,App Login Issue,2450,9.800
5,Interest Query,2440,9.760



Q2 — Issues with Lowest Satisfaction Scores


,issue_type,average_satisfaction,ticket_count
0,Account Statement,2.966451,2474
3,Cheque Bounce,2.969182,2531
9,Wrong Debit,2.972425,2466
5,Interest Query,2.988115,2440
6,KYC Update,2.995986,2491
4,Fraud Report,3.011440,2535
2,Card Blocked,3.012109,2560
8,Net Banking Issue,3.033175,2532
7,Loan Query,3.035700,2521
1,App Login Issue,3.037551,2450



Q3 — Support Issues with Longest Resolution Time


,issue_type,average_resolution_days,ticket_count
3,Cheque Bounce,9.642829,2531
1,App Login Issue,9.599184,2450
9,Wrong Debit,9.558800,2466
6,KYC Update,9.533922,2491
4,Fraud Report,9.483235,2535
5,Interest Query,9.465984,2440
8,Net Banking Issue,9.458531,2532
0,Account Statement,9.420372,2474
2,Card Blocked,9.382812,2560
7,Loan Query,9.363745,2521



Q4 — Resolution Time vs Satisfaction


,issue_type,average_resolution_days,average_satisfaction,ticket_count
3,Cheque Bounce,9.642829,2.969182,2531
1,App Login Issue,9.599184,3.037551,2450
9,Wrong Debit,9.558800,2.972425,2466
6,KYC Update,9.533922,2.995986,2491
4,Fraud Report,9.483235,3.011440,2535
5,Interest Query,9.465984,2.988115,2440
8,Net Banking Issue,9.458531,3.033175,2532
0,Account Statement,9.420372,2.966451,2474
2,Card Blocked,9.382812,3.012109,2560
7,Loan Query,9.363745,3.035700,2521


Correlation between resolution time and satisfaction: -0.304
